# Parameters

In [ ]:
from IPython.display import display
import geopandas as gpd
import pandas as pd
from pyproj import Transformer
from shapely import wkt
import re
from scipy.spatial import cKDTree
# VOLLEDIGE weergave van pandas DataFrames - NOOIT afkorten
pd.set_option('display.max_rows', 1000)           # Alle rijen tonen
pd.set_option('display.max_columns', 1000)        # Alle kolommen tonen
pd.set_option('display.width', None
              )              # Geen breedte limiet
pd.set_option('display.max_colwidth', None)       # Volledige inhoud van cellen
pd.set_option('display.expand_frame_repr', False) # Geen omvouwing naar volgende regel
pd.set_option('display.float_format', '{:.6f}'.format)  # 6 decimalen voor floats

# Ook voor IPython/Jupyter notebooks (als je die gebruikt):
pd.set_option('display.max_seq_items', None)
pd.set_option('display.precision', 10)  # Meer decimalen
pd.set_option('display.show_dimensions', True)  # Toon dimensies onderaan

# Optioneel: Voor Scipy/NumPy arrays
import numpy as np
np.set_printoptions(threshold=np.inf, linewidth=np.inf)
from Utils import Timer, Duur, meet_duur,toon_duur, convert_wkt_string_to_gps, batch_transform_geometry_to_gps_string, convert_coords_to_gps

root_data = "Data/"
root_file_nwb = root_data + "nwb-wegen-08_01_2026.gpkg"
root_bochten = 'Data/Bochten/Bochten/'
wvk = 'wvk_id'
wegvak_columns = ['wvk_id', 'rijrichtng', 'wegnummer', 'wegnr_hmp', 'wegnr_aw', 'geometry',
                      'beginkm', 'eindkm', 'wegbehnaam', 'distrnaam']

df_columns_wegvakken_rename = {'geometry': 'wegvak_geometry',
                               'beginkm': 'wegvak_beginkm',
                               'eindkm': 'wegvak_eindkm'}
df_columns_hectopunten_rename = {'geometry': 'hectopunt_geometry'}
wegvak_columns_dtypes = {
    'wvk_id': 'int',
    'rijrichtng': 'string',
    'wegnummer': 'string',
    'wegnr_hmp': 'string',
    'wegnr_aw': 'string',
    'wegvak_geometry': 'string',
    'wegvak_beginkm': 'float',
    'wegvak_eindkm': 'float',
    'wegvak_rd_coordinaten': 'string',
    'wegvak_gps_coordinaten': 'string'
}
df_columns = ['snelwegnummer', 'oplopend', 'hectomtrng', 'hecto_lttr', 'wvk_id',
              'distrnaam','wegvak_geometry_flat', 'wegvak_rd_coordinaten', 'wegvak_gps_coordinaten', 'hectopunt_geometry', 'hectopunt_geometry_flat']

# hectopunten
hectopunten_columns_dtypes = {
    'wvk_id': 'int',
    'hectomtrng': 'int',
    'hecto_lttr': 'string',
    'hectopunt_geometry_flat': 'string',
    'hecto_lttr': 'string'}

# Bouw data set Analyse

In [ ]:
df = pd.DataFrame()
df['meter'] = np.arange(6400, 102500, 20)

# Inladen Wegvakken

In [ ]:
wegvakken_ = toon_duur(gpd.read_file, root_file_nwb, layer='wegvakken')
wegvakken = wegvakken_.copy()
wegvakken_.columns

In [ ]:
wegvakken_.head(1).T

# Data Cleaning

In [ ]:
transformer = Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True)
def get_rd_coords(geom):
    if geom is None:
        return None
    if geom.geom_type == 'LineString':
        coords = list(geom.coords)
    elif geom.geom_type == 'MultiLineString':
        coords = [pt for line in geom.geoms for pt in line.coords]
    else:
        return None
    return ', '.join(f"{x:.2f} {y:.2f}" for x, y in coords)

def get_gps_coords(geom):
    if geom is None:
        return None
    if geom.geom_type == 'LineString':
        coords = list(geom.coords)
    elif geom.geom_type == 'MultiLineString':
        coords = [pt for line in geom.geoms for pt in line.coords]
    else:
        return None
    gps_pts = [transformer.transform(x, y) for x, y in coords]
    return ', '.join(f"{lat:.6f} {lon:.6f}" for lon, lat in gps_pts)

In [ ]:
# formateren
wegvakken = wegvakken_.copy()
wegvak_columns = ['wvk_id', 'rijrichtng', 'wegnummer', 'wegnr_hmp', 'wegnr_aw', 'geometry',
                      'beginkm', 'eindkm', 'wegbehnaam', 'distrnaam']
# filter op A12
wegvakken = wegvakken[wegvakken['wegnr_aw'] == 'RW12'][wegvak_columns].fillna(-1)

wegvakken = wegvakken.rename(columns=df_columns_wegvakken_rename)
wegvakken['oplopend'] = wegvakken['wegvak_beginkm'] < wegvakken['wegvak_eindkm']
wegvakken['snelwegnummer'] = wegvakken['wegnr_aw'].astype(str).str.replace('RW', '', regex=False)
# coordinaten plat slaan
xy = wegvakken['wegvak_geometry'].apply(
    lambda g: ', '.join([
        f"{x}, {y}"
        for line in g.geoms
        for x, y in line.coords
    ])
)
wegvakken['wegvak_geometry_flat']= xy
# Let op: filter op zijde
wegvakken = wegvakken[wegvakken['oplopend'] == True]
wegvakken['wegvak_rd_coordinaten'] = wegvakken['wegvak_geometry'].apply(get_rd_coords)
wegvakken['wegvak_gps_coordinaten'] = wegvakken['wegvak_geometry'].apply(get_gps_coords)

wegvakken['wegvak_beginkm_meters'] = wegvakken['wegvak_beginkm'] * 1000
wegvakken['wegvak_eindkm_meters'] = wegvakken['wegvak_eindkm'] * 1000

wegvakken = wegvakken.astype(wegvak_columns_dtypes)

# reorder
wegvakken = wegvakken[['wvk_id', 'snelwegnummer', 'wegnummer', 'wegnr_hmp', 'wegnr_aw', 'oplopend', 'rijrichtng', 'wegbehnaam', 'distrnaam',
                       'wegvak_beginkm', 'wegvak_eindkm', 'wegvak_beginkm_meters', 'wegvak_eindkm_meters',
                       'wegvak_geometry', 'wegvak_geometry_flat', 'wegvak_rd_coordinaten', 'wegvak_gps_coordinaten']]

wegvakken.columns

In [ ]:
wegvakken.head(1).T

# Inldaden Hectopunten 

In [ ]:
hectopunten_ = toon_duur(gpd.read_file, root_file_nwb, layer='hectopunten')
hectopunten_.columns

In [ ]:
# formateren
hectopunten = hectopunten_[['hectomtrng', 'afstand', 'wvk_id', 'hecto_lttr', 'geometry']].copy()
hectopunten = hectopunten.sort_values(by=wvk, ascending=True).reset_index()
hectopunten = hectopunten.rename(columns=df_columns_hectopunten_rename)
xy = hectopunten['hectopunt_geometry'].apply(
    lambda g: f"{repr(g.geoms[0].x)}, {repr(g.geoms[0].y)}"
)
hectopunten['hectopunt_geometry_flat'] = xy
hectopunten = hectopunten.astype(hectopunten_columns_dtypes)
hectopunten['meter'] = hectopunten['hectomtrng'] * 1000
hectopunten.head(1).T

# koppelen wegvakken aan hectopunten
let op!: kan alleen op zelfde snelwegnummer ivm overlappende hectopunten voor meerdere snelwegen.

In [ ]:
wegvakken.shape, hectopunten.shape

In [ ]:
hectopunten = hectopunten.merge(
    wegvakken, 
    on='wvk_id',
    how='inner'
)
hectopunten.shape

# Data cleaning
Deze bewerkingen kunnen beter achteraf gedaan worden.

In [ ]:
def get_rd_coords_hectopunt(geom):
    if geom is None:
        return None
    if geom.geom_type == 'Point':
        return f"{geom.x:.2f}, {geom.y:.2f}"
    elif geom.geom_type == 'MultiPoint':
        return f"{geom.geoms[0].x:.2f}, {geom.geoms[0].y:.2f}"
    else:
        return None

def get_gps_coords_hectopunt(geom):
    if geom is None:
        return None
    transformer = Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True)
    if geom.geom_type == 'Point':
        x, y = geom.x, geom.y
    elif geom.geom_type == 'MultiPoint':
        x, y = geom.geoms[0].x, geom.geoms[0].y
    else:
        return None
    lon, lat = transformer.transform(x, y)
    return f"{lat:.6f}, {lon:.6f}"

In [ ]:
hectopunten = hectopunten.sort_values(by='hectomtrng', ascending=True)
hectopunten['hectopunt_rd_coordinaten'] = hectopunten['hectopunt_geometry'].apply(get_rd_coords_hectopunt)
hectopunten['hectopunt_gps_coordinaten'] = hectopunten['hectopunt_geometry'].apply(get_gps_coords_hectopunt)
hectopunten = hectopunten[['wvk_id', 'snelwegnummer', 'wegnummer', 'wegnr_hmp', 'wegnr_aw', 'oplopend', 'rijrichtng', 'wegbehnaam', 'distrnaam', 'hecto_lttr',
                           'meter', 'hectomtrng', 'wegvak_beginkm', 'wegvak_eindkm', 'wegvak_beginkm_meters', 'wegvak_eindkm_meters',
                           'hectopunt_geometry', 'hectopunt_geometry_flat', 'hectopunt_rd_coordinaten', 'hectopunt_gps_coordinaten',
                           'wegvak_geometry', 'wegvak_geometry_flat', 'wegvak_rd_coordinaten', 'wegvak_gps_coordinaten']]
hectopunten.head(1).T

# Tussentijdse opslag

In [ ]:
hectopunten.to_csv('Analyse-data_set.csv')
# hectopunten.columns

In [ ]:
hectopunten = pd.read_csv('Analyse-data_set.csv')
hectopunten.columns

# Inladen Bochten

In [ ]:
bochten = gpd.read_file(root_bochten + "bochten_w.shp")
bochten.columns

In [ ]:
bochten.shape

# Data cleaning

In [ ]:
transformer = Transformer.from_crs('EPSG:28992', 'EPSG:4326', always_xy=True)

def rd_coords_flat(geom):
    if geom is None:
        return None
    
    if geom.geom_type == 'LineString':
        coords = geom.coords
    elif geom.geom_type == 'MultiLineString':
        coords = []
        for line in geom.geoms:
            coords.extend(line.coords)
    else:
        return None

    return ", ".join(f"{round(x,3)}, {round(y,3)}" for x, y in coords)


def gps_coords_flat(geom):
    if geom is None:
        return None
    
    if geom.geom_type == 'LineString':
        coords = geom.coords
    elif geom.geom_type == 'MultiLineString':
        coords = []
        for line in geom.geoms:
            coords.extend(line.coords)
    else:
        return None

    out = []
    for x, y in coords:
        lon, lat = transformer.transform(x, y)
        out.append(f"{round(lat,6)}, {round(lon,6)}")
    
    return ", ".join(out)




In [ ]:
# coordinaten plat slaan
xy = bochten['geometry'].apply(
    lambda g: ', '.join([
        f"{x}, {y}"
        for x, y in g.coords
    ])
)
bochten['bochten_geometry_flat']= xy


bochten['bochten_rd_coordinaten'] = bochten['geometry'].apply(rd_coords_flat)
bochten['bochten_gps_coordinaten'] = bochten['geometry'].apply(gps_coords_flat)

bochten.head(1)

# Analyse
Hieronder bekijk ik hoeveel bochten worden herkend in mijn eigen data set

In [ ]:
# zelfde kolomnamen <- wvk_id
bochten.rename(columns={'WVK_ID': 'wvk_id'}, inplace=True)
bochten['wvk_id'] = bochten['wvk_id'].astype(float)

a = hectopunten['wvk_id'].tolist()

b = [int(x) for x in set(a) if x == x]
c = list(set([int(x) for x in bochten['wvk_id'].tolist() if x == x]))

counter = 0
for i in b:
    for j in c:
        if i == j:
            # print(True, i, j)
            counter += 1
        # break
    # break
counter

Er zijn voldoende overeenkomsten om met de bochtendataset verder te gaan. Het is niet perse nodig om een eigen bochten dataset te creeeren. Door een eigen bochtenstralen dataset te ontwikkelen zouden weliswaar meet bochten kunnen worden ingeladen maar gezien het tijdsbestek kan beter aandacht besteed worden aan het modeleren.

In [ ]:
hectopunten.columns

In [ ]:
bochten.columns

geometrie komt niet overeen dus koppelen o.b.v. meest dichbijzijnde afstand door middel van Eucldische afstand.
# koppelen bochten aan hectopunten

In [ ]:
hectopunten[hectopunten['wvk_id'] == 166309055][['wvk_id', 'meter',]]

In [ ]:
bochten[bochten['wvk_id'] == 166309055][['wvk_id', 'OMSBPS', 'DRAAIHOEK']]

Bram gaf aan dat er een kortere en simpelere route is naar het koppelen van de datasets door middel van de kolom OMSBPS. Echter bevat deze kolom ook rijen die onbekend zijn.
Daarom gaan we toch koppelen op basis van de meest dichtbijzijnde coordinaten punt.

In [ ]:
bochten.isnull().sum()

In [ ]:
bochten.shape

In [ ]:
hectopunten.shape

In [ ]:
bochten.columns

In [ ]:
from scipy.spatial.distance import cdist
import numpy as np
import pandas as pd

# --- 1. Eerste coördinaat uit geometry_flat pakken ---
def parse_first_rd_point(geom_str):
    parts = str(geom_str).split(',')
    return float(parts[0].strip()), float(parts[1].strip())

hectopunten['_rd_x'] = hectopunten['hectopunt_geometry_flat'].apply(lambda g: parse_first_rd_point(g)[0])
hectopunten['_rd_y'] = hectopunten['hectopunt_geometry_flat'].apply(lambda g: parse_first_rd_point(g)[1])

bochten['_rd_x'] = bochten['bochten_geometry_flat'].apply(lambda g: parse_first_rd_point(g)[0])
bochten['_rd_y'] = bochten['bochten_geometry_flat'].apply(lambda g: parse_first_rd_point(g)[1])

# --- 2. Per wvk_id: dichtstbijzijnde bocht, anders NaN ---
matched_indices = []

for i, hecto_row in hectopunten.iterrows():
    wvk = hecto_row['wvk_id']
    bocht_candidates = bochten[bochten['wvk_id'] == wvk]

    if bocht_candidates.empty:
        matched_indices.append(None)
        continue

    hecto_pt  = np.array([[hecto_row['_rd_x'], hecto_row['_rd_y']]])
    bocht_pts = bocht_candidates[['_rd_x', '_rd_y']].values

    distances         = cdist(hecto_pt, bocht_pts)[0]
    nearest_local_idx = np.argmin(distances)
    nearest_distance  = distances[nearest_local_idx]

    if nearest_distance > 250:
        matched_indices.append(None)
    else:
        matched_indices.append(bocht_candidates.iloc[nearest_local_idx].name)

# --- 3. Kolommen toevoegen ---
valid_mask = [idx is not None for idx in matched_indices]
bochten_cols = ['DRAAIHOEK', 'BOOGSTRAAL', 'geometry', 'bochten_geometry_flat', 'bochten_rd_coordinaten', 'bochten_gps_coordinaten']

bochten_matched = pd.DataFrame(index=range(len(hectopunten)), columns=bochten_cols)
for pos, idx in enumerate(matched_indices):
    if idx is not None:
        bochten_matched.loc[pos, bochten_cols] = bochten.loc[idx, bochten_cols].values

df_merged = pd.concat([hectopunten.reset_index(drop=True), bochten_matched], axis=1)

# --- 4. Opschonen ---
df_merged.drop(columns=['_rd_x', '_rd_y'], inplace=True)
hectopunten.drop(columns=['_rd_x', '_rd_y'], inplace=True)
bochten.drop(columns=['_rd_x', '_rd_y'], inplace=True)


hectopunten = df_merged.copy()
hectopunten = hectopunten.rename(columns={'DRAAIHOEK': 'draaihoek', 'BOOGSTRAAL': 'boogstraal', 'geometry': 'bochten_geometry'})
hectopunten = hectopunten[['wvk_id', 'snelwegnummer', 'wegnummer', 'wegnr_hmp', 'wegnr_aw', 'oplopend', 'rijrichtng', 'wegbehnaam', 'distrnaam', 'hecto_lttr',
                           'meter', 'hectomtrng', 'wegvak_beginkm', 'wegvak_eindkm', 'wegvak_beginkm_meters', 'wegvak_eindkm_meters',
                           'draaihoek', 'boogstraal',
                           'hectopunt_geometry', 'hectopunt_geometry_flat', 'hectopunt_rd_coordinaten', 'hectopunt_gps_coordinaten',
                           'bochten_geometry', 'bochten_rd_coordinaten', 'bochten_gps_coordinaten',
                           'wegvak_geometry', 'wegvak_geometry_flat', 'wegvak_rd_coordinaten', 'wegvak_gps_coordinaten']]

print(f"Hectopunten: {len(hectopunten)} → Merged: {len(df_merged)}")
print(f"Geen wvk_id match: {sum(1 for x in matched_indices if x is None)} rijen leeg")

In [ ]:
hectopunten.columns

In [ ]:
hectopunten.shape

In [ ]:
hectopunten[hectopunten['wvk_id'] == 166309055].T

# Tussentijdse opslag

In [ ]:
hectopunten.to_csv('Analyse-data_set.csv')

In [ ]:
hectopunten = pd.read_csv('Analyse-data_set.csv')
hectopunten.columns

# Koppelen hectopunten aan Analyse data set

# Inladen Inspectigence

In [ ]:
insp = pd.read_csv("Data/Inspectigence/Inspectigence-A122023/csv/Inspectigence-A122023_selected.csv")
insp.head()

# Data cleaning

In [ ]:
from shapely import wkt
from shapely.geometry.base import BaseGeometry

def parse_geom(g):
    if isinstance(g, BaseGeometry):
        return g
    return wkt.loads(g)

insp['Geometry'] = insp['Geometry'].apply(parse_geom)

insp['Geometry_flat'] = insp['Geometry'].apply(
    lambda g: ', '.join([f"{x}, {y}" for x, y in g.coords])
)

from pyproj import Transformer
import re

transformer_to_rd = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)

def parse_linestring_all_points(geom_str):
    """Haal alle (lon, lat) paren uit een LINESTRING."""
    nums = re.findall(r'[\d.]+', str(geom_str))
    coords = [(float(nums[i]), float(nums[i+1])) for i in range(0, len(nums)-1, 2)]
    return coords

def linestring_to_rd(geom_str):
    """LINESTRING GPS → RD coördinaten string (zoals wegvak_rd_coordinaten)."""
    coords = parse_linestring_all_points(geom_str)
    rd_pairs = []
    for lon, lat in coords:
        x, y = transformer_to_rd.transform(lon, lat)
        rd_pairs.append(f"{x:.2f} {y:.2f}")
    return ', '.join(rd_pairs)

def linestring_to_gps(geom_str):
    """LINESTRING GPS → nette GPS coördinaten string (zoals wegvak_gps_coordinaten)."""
    coords = parse_linestring_all_points(geom_str)
    gps_pairs = []
    for lon, lat in coords:
        gps_pairs.append(f"{lat:.9f} {lon:.9f}")
    return ', '.join(gps_pairs)

# Toepassen op insp dataframe
insp['Geometry_rd_coordinaten']  = insp['Geometry'].apply(linestring_to_rd)
insp['Geometry_gps_coordinaten'] = insp['Geometry'].apply(linestring_to_gps)

In [ ]:
insp.sample(1)

# Analyse

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 1, figsize=(12, 3))

for ax, col in zip(axes, ['Health_Score', 'Visibility_Score']):
    scores = insp[col].dropna().values
    ax.plot(scores, np.zeros_like(scores), '|', alpha=0.3, color='steelblue', markersize=10)
    ax.set_yticks([])
    ax.set_ylabel(col, rotation=0, labelpad=100, va='center')
    ax.spines[['left', 'top', 'right']].set_visible(False)

axes[1].set_xlabel('Score')
plt.tight_layout()
plt.show()

In [ ]:
insp = insp.sort_values(by='Health_Score', ascending=True)

In [ ]:
insp.head(100)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
from scipy.stats import gaussian_kde
from matplotlib.patches import FancyArrowPatch

cols = ['Health_Score', 'Visibility_Score']

# Drempelwaarden — pas aan op jouw domein
THRESHOLDS = {'slecht': 4, 'matig': 7}   # <4 = slecht, 4-7 = matig, >7 = goed

score_cmap = mcolors.LinearSegmentedColormap.from_list(
    'rg', ['#d73027', '#fee08b', '#1a9850']   # rood → geel → groen
)

fig, axes = plt.subplots(len(cols), 1, figsize=(13, 3.2 * len(cols)),
                         gridspec_kw={'hspace': 0.75})

for ax, col in zip(axes, cols):
    data = insp[col].dropna().values
    xmin, xmax = data.min(), data.max()

    # --- KDE ---
    kde  = gaussian_kde(data, bw_method=0.12)
    x    = np.linspace(xmin, xmax, 600)
    dens = kde(x)

    # --- Gekleurde KDE: elke strip-segment krijgt kleur van zijn x-waarde ---
    norm_score = mcolors.Normalize(vmin=xmin, vmax=xmax)
    for i in range(len(x) - 1):
        ax.fill_between(x[i:i+2], dens[i:i+2], alpha=0.85,
                        color=score_cmap(norm_score(x[i])))

    ax.plot(x, dens, color='#333', lw=0.8, alpha=0.5)

    # --- Drempellijnen + zone-labels ---
    for thresh, label in THRESHOLDS.items():
        ax.axvline(label, color='#333', lw=1.2, ls='--', zorder=5)
        ax.text(label + 0.08, dens.max() * 1.05, f'{label}', fontsize=8,
                color='#333', va='bottom')

    # --- % per zone berekenen + annoteren ---
    t1, t2 = THRESHOLDS['slecht'], THRESHOLDS['matig']
    pct_bad  = (data < t1).mean() * 100
    pct_mid  = ((data >= t1) & (data < t2)).mean() * 100
    pct_good = (data >= t2).mean() * 100

    zones = [
        (xmin,  t1,  pct_bad,  '#d73027', 'Slecht'),
        (t1,    t2,  pct_mid,  '#d9a02b', 'Matig'),
        (t2,    xmax,pct_good, '#1a9850', 'Goed'),
    ]
    for z_min, z_max, pct, color, lbl in zones:
        mid = (z_min + z_max) / 2
        ax.text(mid, dens.max() * 1.22, f'{lbl}\n{pct:.1f}%',
                ha='center', va='bottom', fontsize=9.5,
                fontweight='bold', color=color)

    # --- Mediaan ---
    med = np.median(data)
    ax.axvline(med, color='#111', lw=1.8, zorder=6)
    ax.text(med, dens.max() * 0.6, f' mediaan\n {med:.2f}',
            fontsize=8.5, color='#111', va='center')

    ax.set_xlim(xmin - 0.1, xmax + 0.1)
    ax.set_ylim(0, dens.max() * 1.55)
    ax.set_title(col, fontsize=13, fontweight='bold', loc='left', pad=8)
    ax.set_xlabel('Score', fontsize=10)
    ax.set_yticks([])
    ax.spines[['top', 'right', 'left']].set_visible(False)

plt.suptitle('Scoreverdeling — rood = slecht, groen = goed', fontsize=13,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
CUTOFF = 9  # alles >= 9 weghakken

fig, axes = plt.subplots(len(cols), 1, figsize=(13, 3.2 * len(cols)),
                         gridspec_kw={'hspace': 0.75})

for ax, col in zip(axes, cols):
    data_full = insp[col].dropna().values
    data      = data_full[data_full < CUTOFF]   # ← ingezoomd

    pct_shown = (data_full < CUTOFF).mean() * 100

    xmin, xmax = data.min(), data.max()

    kde  = gaussian_kde(data, bw_method=0.12)
    x    = np.linspace(xmin, xmax, 600)
    dens = kde(x)

    norm_score = mcolors.Normalize(vmin=xmin, vmax=xmax)
    for i in range(len(x) - 1):
        ax.fill_between(x[i:i+2], dens[i:i+2], alpha=0.85,
                        color=score_cmap(norm_score(x[i])))
    ax.plot(x, dens, color='#333', lw=0.8, alpha=0.5)

    for thresh, label in THRESHOLDS.items():
        if xmin < label < xmax:
            ax.axvline(label, color='#333', lw=1.2, ls='--', zorder=5)
            ax.text(label + 0.08, dens.max() * 1.05, f'{label}',
                    fontsize=8, color='#333', va='bottom')

    t1, t2 = THRESHOLDS['slecht'], THRESHOLDS['matig']
    zones = [
        (xmin,             min(t1, xmax),  '#d73027', 'Slecht'),
        (max(xmin, t1),    min(t2, xmax),  '#d9a02b', 'Matig'),
        (max(xmin, t2),    xmax,           '#1a9850', 'Goed'),
    ]
    for z_min, z_max, color, lbl in zones:
        if z_max > z_min:
            pct = ((data >= z_min) & (data < z_max)).mean() * 100
            mid = (z_min + z_max) / 2
            ax.text(mid, dens.max() * 1.22, f'{lbl}\n{pct:.1f}%',
                    ha='center', va='bottom', fontsize=9.5,
                    fontweight='bold', color=color)

    med = np.median(data)
    ax.axvline(med, color='#111', lw=1.8, zorder=6)
    ax.text(med, dens.max() * 0.6, f' mediaan\n {med:.2f}',
            fontsize=8.5, color='#111', va='center')

    # --- afgekapte staart annotatie ---
    ax.annotate(f'score ≥ {CUTOFF} niet getoond\n({100 - pct_shown:.1f}% van data)',
                xy=(xmax, dens.max() * 0.15),
                fontsize=8.5, color='gray', ha='right',
                bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#ccc', lw=0.8))

    ax.set_xlim(xmin - 0.1, xmax + 0.2)
    ax.set_ylim(0, dens.max() * 1.55)
    ax.set_title(col, fontsize=13, fontweight='bold', loc='left', pad=8)
    ax.set_xlabel('Score  (ingezoomd op < 9)', fontsize=10)
    ax.set_yticks([])
    ax.spines[['top', 'right', 'left']].set_visible(False)

plt.suptitle(f'Slechte scores uitgelicht  (< {CUTOFF})',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Koppelen Inspecitence aan hectopunten

In [ ]:
hectopunten.shape

In [ ]:
# --- 0. Coördinaten parser ---
def parse_multipoint_rd(geom_str):
    nums = re.findall(r'[\d.]+', str(geom_str))
    return float(nums[0]), float(nums[1])

def parse_linestring_gps(geom_str):
    nums = re.findall(r'[\d.]+', str(geom_str))
    return float(nums[0]), float(nums[1])

# --- 1. Transformeer GPS → RD ---
transformer = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)

hecto_coords = hectopunten['hectopunt_geometry'].apply(parse_multipoint_rd)
hecto_xy = np.array(hecto_coords.tolist())

def gps_to_rd(geom_str):
    lon, lat = parse_linestring_gps(geom_str)
    x, y = transformer.transform(lon, lat)
    return x, y

insp_coords = insp['Geometry'].apply(gps_to_rd)
insp_xy = np.array(insp_coords.tolist())

# --- 2. KDTree koppeling ---
tree = cKDTree(hecto_xy)
distances, indices = tree.query(insp_xy, k=1)

# --- 3. Voeg hecto_idx en afstand toe aan insp ---
insp_matched = insp.copy().reset_index(drop=True)
insp_matched['_hecto_idx'] = indices
insp_matched['_dist_m']    = distances

# --- 4. Groepeer ALLE insp-kolommen per hectopunt ---
def join_values(series):
    return ', '.join(series.dropna().astype(str).tolist())

all_insp_cols = [col for col in insp_matched.columns if col != '_hecto_idx']
agg_dict = {col: join_values for col in all_insp_cols}

insp_grouped = (
    insp_matched
    .groupby('_hecto_idx', as_index=False)
    .agg(agg_dict)
)

# Aantal matches toevoegen
match_counts = insp_matched.groupby('_hecto_idx').size().reset_index(name='_n_matches')
insp_grouped = insp_grouped.merge(match_counts, on='_hecto_idx', how='left')

# Prefix insp_ op alle insp-kolommen
insp_grouped = insp_grouped.rename(columns={
    col: f'insp_{col}'
    for col in insp_grouped.columns
    if col != '_hecto_idx'
})

# --- 5. Hectopunten als basis, ALLE kolommen behouden ---
hectopunten_base = hectopunten.copy().reset_index(drop=True)
hectopunten_base['_hecto_idx'] = hectopunten_base.index

df_final = hectopunten_base.merge(
    insp_grouped,
    on='_hecto_idx',
    how='left'
).drop(columns=['_hecto_idx'])
hectopunten = df_final.copy()

# --- 6. Prints ---
print("=" * 60)
print("KOPPELINGSRESULTAAT")
print("=" * 60)
print(f"Hectopunten totaal      : {len(hectopunten):>8,}")
print(f"Inspecties totaal       : {len(insp):>8,}")
print(f"Hectopunten met match   : {df_final['insp__n_matches'].notna().sum():>8,}")
print(f"Hectopunten zonder match: {df_final['insp__n_matches'].isna().sum():>8,}")
print(f"Uitvoer rijen           : {len(df_final):>8,}")
print()

first_dist = df_final['insp__dist_m'].dropna().apply(lambda x: float(str(x).split(',')[0]))
print("AFSTANDSSTATISTIEKEN (meter)")
print(f"  Gemiddeld  : {first_dist.mean():>8.1f} m")
print(f"  Mediaan    : {first_dist.median():>8.1f} m")
print(f"  P90        : {first_dist.quantile(0.90):>8.1f} m")
print(f"  Max        : {first_dist.max():>8.1f} m")
print()

# Afstandsbuckets
buckets = [25, 50, 100, 250, np.inf]
labels  = ['≤ 25m', '25–50m', '50–100m', '100–250m', '> 250m']
print("VERDELING AFSTAND")
for low, high, lbl in zip([0] + buckets[:-1], buckets, labels):
    n   = ((first_dist >= low) & (first_dist < high)).sum()
    pct = n / len(first_dist) * 100
    bar = '█' * int(pct / 2)
    print(f"  {lbl:<12} {n:>6,}  ({pct:5.1f}%)  {bar}")
print()

# Verdeling aantal matches per hectopunt
print("VERDELING AANTAL MATCHES PER HECTOPUNT")
counts = df_final['insp__n_matches'].dropna().astype(int).value_counts().sort_index()
max_count = counts.max()
for n, cnt in counts.items():
    bar = '█' * int(cnt / max_count * 20)
    print(f"  {n} match(es): {cnt:>6,}  {bar}")
print()

# Waarschuwing slechte koppelingen
THRESHOLD_M = 250
bad = (first_dist > THRESHOLD_M).sum()
if bad > 0:
    print(f"⚠️  {bad} rijen ({bad/len(first_dist)*100:.1f}%) hebben afstand > {THRESHOLD_M}m")
else:
    print(f"✓  Alle koppelingen binnen {THRESHOLD_M}m")
print()

print(f"TOTAAL KOLOMMEN: {len(df_final.columns)}")
print("Hectopunt-kolommen:", [c for c in df_final.columns if not c.startswith('insp_')])
print("Insp-kolommen     :", [c for c in df_final.columns if c.startswith('insp_')])

In [ ]:
hectopunten.shape

In [ ]:
hectopunten.sample(1).T

In [ ]:
hectopunten.columns

# Tussentijdse opslag

In [ ]:
hectopunten.to_csv('Analyse-data_set.csv')

In [ ]:
hectopunten = pd.read_csv('Analyse-data_set.csv')
hectopunten.columns

# Tests

In [ ]:
hectopunten.sample(1).T